# Lab 1 (Week 2) -- worked examples

This notebook walks through the example code in `code/lab1/lab_1.py`, which is
the file the lab sheet points you at when it says *"there is an example in the
repository if you need help"*.

It is **not** the lab solution. It shows you

* Task 1: the pure Python Euclidean distance, and the timing/plotting pattern you
  need for the timing experiment.
* Task 2: how to load the two data files in `data/lab1`.

The tasks themselves -- the Numba version, DTW, the preprocessing pipeline and
the single feature classifier -- are left for you. There are empty cells below to
write them in.

The lab sheet is `Lab Sheet 1.pdf`, and there is a summary in the
repository `README.md`. If you want NumPy practice first, work through
`labW1_Intro_to_Jupyter.ipynb`.

## Setup

`lab_1.py` lives in `code/lab1`, which is not on the import path for this
notebook, so we add it before importing. Everything else is the standard
scientific Python stack.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "code" / "lab1"))

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_1

print("repo root:", REPO_ROOT)
print("data dir :", lab_1.DATA_DIR)

repo root: C:\Code\COMP3222-2026
data dir : C:\Code\COMP3222-2026\data\lab1


---

## Task 1: Distance functions and Numba

Measuring distance is fundamental to many ML algorithms. Euclidean distance
between two series of length $m$ is

$$d_e(a, b) = \sqrt{\sum_{i=1}^{m}(a_i - b_i)^2}$$

The lab sheet asks for a single loop, with no input checks -- both series are
assumed to be 1D NumPy arrays of the same length.

In [5]:
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
y = np.array([2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0])

print("ours   :", lab_1.euclidean_distance_python(x, y))
print("numpy  :", np.linalg.norm(x - y))   # sanity check, sqrt(10) = 3.1622...

ours   : 3.1622776601683795
numpy  : 3.1622776601683795


### Task 1.2: the Numba version

Now write the second version yourself. Copy the function above, give it a
different name, and decorate it with `@njit(cache=True)` (see lecture 1.2, slide
40). The body does not change at all -- that is the point.

```python
from numba import njit

@njit(cache=True)
def euclidean_distance_numba(x, y):
    ...
```

`njit` compiles the function to machine code the first time it is called with a
given argument type. `cache=True` writes that compiled code to disk (a
`__pycache__` folder next to the source) so later *runs of the program* do not
pay the compilation cost again.

There is a trap waiting for you in the next section: **the first call is slow**,
because it includes compilation. If you time the first call you will be timing
the compiler, not your code. Call the function once on small inputs before you
start timing, and check for yourself how much difference that makes.

In [ ]:
# Your code: the @njit version of euclidean_distance_python.

In [ ]:
# Your code: time the first call and the second call of your Numba
# function on the same small input. How much of the first call was compilation?

### Timing experiment

`time_distance` generates two random series of length `n` and times a single
call. The pattern is the one from the lab sheet: read the clock, call the
function, read the clock again.

These functions are fast, so `n` has to be large before the Python version takes
a measurable amount of time. We go up to a million points.

In [ ]:
print(inspect.getsource(lab_1.time_distance))

In [ ]:
sizes, t_py = [], []
for n in range(100_000, 1_000_001, 100_000):
    sizes.append(n)
    t_py.append(lab_1.time_distance(lab_1.euclidean_distance_python, n))
    print(f"n = {n:>9,}  python = {t_py[-1]:.6f}s")

In [ ]:
# Your code: time your Numba version over the same sizes into a list t_nb,
# remembering to warm it up first, then estimate the speed-up.

### Plotting time against n

Both axes are on a log scale. On log--log axes a function whose runtime is
$O(n^k)$ appears as a straight line of gradient $k$, so this should come out
roughly straight with gradient 1 -- it is $O(n)$.

Add your Numba timings as a second line. The two should be parallel, separated
vertically by the speed-up.

In [ ]:
plt.figure(figsize=(7.5, 5.0))
plt.plot(sizes, t_py, marker="o", label="ED Python")
# Your code: plot your Numba timings as a second line.
plt.xscale("log")
plt.yscale("log")
plt.xlabel("n (log scale)")
plt.ylabel("Time (seconds, log scale)")
plt.title("Euclidean distance: time vs n")
plt.grid(True, which="both", linestyle="--", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

`lab_1.part1_example()` does all of the above in one call and also saves the
figure to `jit_speed_time_vs_n.png`. Running `python code/lab1/lab_1.py` from a
terminal runs it too.

### Optional extension: Dynamic Time Warping

Do tasks 2 and 3 first, then come back here.

DTW is an $O(n^2)$ time series distance that can compensate for misalignment
between two series. Build an $(m+1) \times (m+1)$ cost matrix `C` filled with
infinity, set `C[0, 0] = 0`, then for each `i, j` in `1...m`

```
C[i, j] = (x[i-1] - y[j-1])**2 + min(C[i-1, j-1], C[i-1, j], C[i, j-1])
```

and return `C[m, m]`. Write a plain Python version and an `@njit` version, then
re-run the timing experiment above with much smaller values of `n` -- it is
quadratic, so a million points is far too many -- and estimate the speed-up for
both Euclidean and DTW.

The hint from the lab sheet:

```python
n = len(x)
C = np.full((n + 1, n + 1), np.inf, dtype=np.float64)
C[0, 0] = 0.0
```

In [ ]:
def dtw_distance_python(x: np.ndarray, y: np.ndarray) -> float:
    """Your DTW implementation goes here."""
    ...

---

## Task 2: scikit-learn preprocessing and pipelines

Lecture 2 covered three things that often need doing before a model sees the
data: encoding categorical variables, filling in missing values, and putting
features on a comparable scale. `lab_1.py` shows you how to get the two data
files into memory; the preprocessing is yours to write.

### A. Categorical variables -- `one_hot.csv`

Six columns: three continuous (`x1, x2, x3`), one ordinal (`rank`), one nominal
(`colour`), and the class label `y` last. There is no header row, so we pass
`header=None` and supply the names ourselves.

In [ ]:
df = pd.read_csv(lab_1.DATA_DIR / "one_hot.csv", header=None,
                 names=["x1", "x2", "x3", "rank", "colour", "y"])
df.head()

In [ ]:
X_cont, rank, colour, y = lab_1.one_hot_example()

`rank` is ordinal, so its numeric values are meaningful and it can go straight
into the feature matrix as a float column (`np.column_stack`). `colour` is
nominal -- RED is not "more" than BLUE -- so it needs one-hot encoding with
`sklearn.preprocessing.OneHotEncoder`. Note that the encoder expects 2D input,
so you will have to reshape `colour`.

Then sanity check: can you fit a classifier on the result?

In [ ]:
# Your code: column_stack the continuous features with rank,
# one-hot encode colour, and fit any classifier on the result.

### B. Missing values -- `missing.csv`

Four columns this time: three continuous features and the label. Missing values
appear either as a blank cell or as the text `NaN`; `pd.read_csv` reads both as
`np.nan`.

In [ ]:
X_missing, y_missing = lab_1.missing_example()
pd.DataFrame(X_missing, columns=["x1", "x2", "x3"]).head(8)

In [ ]:
# Missing values per column
pd.DataFrame(X_missing, columns=["x1", "x2", "x3"]).isna().sum()

Now explore `sklearn.impute` and use one of them -- `SimpleImputer` is the
obvious starting point -- to fill in the gaps.

In [ ]:
# Your code: impute the missing values.

### C. Scaling / standardisation

`StandardScaler` from `sklearn.preprocessing` centres each feature on zero and
scales it to unit variance. Apply it to the imputed data.

In [ ]:
# Your code: standardise the imputed data.

### D. Pipelines

A `Pipeline` chains the preprocessing steps and the model into a single
estimator, so `fit` and `predict` apply the whole sequence. This matters for
correctness as well as tidiness -- the scaler is fitted on the training data
only, which is what you want.

```python
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", KNeighborsClassifier()),
])
```

To go further, look at `ColumnTransformer` for the `one_hot` data, which mixes
categorical and continuous columns and so needs different treatment per
column.

In [ ]:
# Your code: build the pipeline.

### E. Train/test split

Always evaluate on data the model has not seen.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
```

There is much more on evaluation in Week 5.

In [ ]:
# Your code: split, fit and predict.

---

## Task 3: Implement your own classifier

Write a classifier that follows the scikit-learn conventions, so it works with
`Pipeline`, `train_test_split` and everything else later in the module. It uses
a decision rule on a single feature, and only handles binary problems.

* **Constructor** -- store the index of the feature to use (default 0). By the
  scikit-learn convention the constructor only stores its arguments; it does no
  work and reads no data.
* **`fit`** -- take the chosen column, find its mean for each class, and set the
  threshold to the median of all values in that column. Whichever class has the
  lower mean becomes the "less than" prediction. Attributes learned in `fit` end
  with an underscore, e.g. `self.threshold_`.
* **`predict`** -- apply the rule to the chosen column of each case. If the
  class 0 mean was the smaller one, predict class 0 when `value < threshold`,
  otherwise class 1.

Inheriting from `BaseEstimator` and `ClassifierMixin` gets you `get_params`,
`set_params` and `score` for free.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin


class SingleFeatureClassifier(ClassifierMixin, BaseEstimator):
    """Classify on a threshold applied to one feature."""

    def __init__(self, feature_index: int = 0):
        self.feature_index = feature_index

    def fit(self, X, y):
        # Your code: class means for the chosen column, the median threshold,
        # and which side of the threshold is class 0.
        ...
        return self

    def predict(self, X):
        # Your code: apply the rule.
        ...

In [ ]:
# Your code: test it. Fit on some data and check the accuracy is
# better than guessing.

If you finish early, explore the range of classifiers in scikit-learn -- we
will be using them next week.